In [346]:
x = '4D'  # No of days of interval
y = 30 #no of predictions

In [347]:
open_preds = []
high_preds = []
low_preds = []
close_preds = []
volume_preds = []
turnover_preds = [] 

In [348]:
# ============================================================
# 1️⃣ IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import seaborn as sns
warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, precision_score, recall_score, f1_score, accuracy_score, confusion_matrix

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, SimpleRNN
from tensorflow.keras.callbacks import EarlyStopping

from tabulate import tabulate

## for close price

In [349]:
# ============================================================
# 2️⃣ LOAD & PREPROCESS MULTI-YEAR NSE DATASET
# ============================================================

df = pd.read_csv("NIFTY_5_Years.csv", encoding="utf-8-sig")

# Clean columns
df.columns = df.columns.str.strip()

# Convert date
df['Date'] = pd.to_datetime(df['Date'])

# Sort ascending
df = df.sort_values("Date")

# Set index
df.set_index("Date", inplace=True)

# Use Close price
data = df[['Close']]

# Train-Test Split (80-20)
train_size = int(len(data) * 0.8)
train, test = data[:train_size], data[train_size:]

print("Training size:", len(train))
print("Testing size :", len(test))

Training size: 992
Testing size : 248


In [350]:
# ============================================================
# 4️⃣ HELPER FUNCTIONS
# ============================================================

def create_sequences(data, time_steps=60):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:i+time_steps])
        y.append(data[i+time_steps])
    return np.array(X), np.array(y)


def evaluate_model(actual, predicted, name):
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    r2 = r2_score(actual, predicted)

    print(f"\n{name}")
    print("RMSE:", rmse)
    print("MAE :", mae)
    print("R2  :", r2)

    return rmse, mae, r2


time_steps = 60

In [351]:
import numpy as np

def predict_future_prices(model, last_data, scaler, time_steps, future_days):
    
    # Ensure correct shape
    input_seq = last_data[-time_steps:]
    input_seq = input_seq.reshape(1, time_steps, 1)

    future_predictions = []

    for _ in range(future_days):
        
        pred = model.predict(input_seq, verbose=0)
        
        # Store prediction
        future_predictions.append(pred[0, 0])
        
        # Reshape prediction properly to (1,1,1)
        pred_reshaped = pred.reshape(1, 1, 1)
        
        # Remove first timestep and append prediction
        input_seq = np.concatenate(
            (input_seq[:, 1:, :], pred_reshaped),
            axis=1
        )

    # Convert back to original scale
    future_predictions = scaler.inverse_transform(
        np.array(future_predictions).reshape(-1, 1)
    )

    return future_predictions.flatten()

In [352]:
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

In [353]:
time_steps = 30

# --- SARIMA ---
sarima_model = SARIMAX(train['Close'],
                       order=(4,0,3),
                       seasonal_order=(1,0,1,5))

sarima_result = sarima_model.fit()

forecast = sarima_result.forecast(steps=len(test))

# --- Residuals ---
residuals = test['Close'].values - forecast.values
residuals = residuals.reshape(-1,1)

# --- Scaling ---
from sklearn.preprocessing import StandardScaler
scaler_res = StandardScaler()
res_scaled = scaler_res.fit_transform(residuals)

X_res, y_res = create_sequences(res_scaled, time_steps)

# --- RNN ---
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

model = Sequential([
    SimpleRNN(61, return_sequences=True, input_shape=(time_steps,1)),
    Dropout(0.2),
    SimpleRNN(32),
    Dense(1)
])

model.compile(optimizer=Adam(0.0005), loss='huber')

early_stop = EarlyStopping(patience=10, restore_best_weights=True)

model.fit(X_res, y_res,
          epochs=150,
          batch_size=16,
          callbacks=[early_stop],
          verbose=0)

# --- Prediction ---
pred = model.predict(X_res)
pred = scaler_res.inverse_transform(pred)

final_pred = forecast[time_steps:].values + pred.flatten()
actual = test['Close'].values[time_steps:]

7/7 [==============================] - 0s 3ms/step


In [354]:
future_days = y
future_prices = predict_future_prices(
    model,
    test_scaled,
    scaler,
    time_steps,
    future_days
)

print("Future Prices:", future_prices)

Future Prices: [25643.445 25762.727 25770.672 25703.72  25872.717 26046.424 26119.713
 26032.21  25963.14  26037.424 26139.557 26097.412 26097.715 26103.21
 26221.367 26233.924 26185.479 26053.662 26042.697 26144.277 26140.973
 26065.414 25970.686 25975.07  26017.727 25985.625 25942.387 25943.137
 26048.365 26144.627]


In [355]:
last_date = df.index[-1]
future_dates = pd.date_range(last_date, periods=future_days+1, freq=x)[1:]

future_df = pd.DataFrame({
    "Date": future_dates,
    "Predicted_Price": future_prices
})

print(future_df)

         Date  Predicted_Price
0  2026-02-24     25643.445312
1  2026-02-28     25762.726562
2  2026-03-04     25770.671875
3  2026-03-08     25703.720703
4  2026-03-12     25872.716797
5  2026-03-16     26046.423828
6  2026-03-20     26119.712891
7  2026-03-24     26032.210938
8  2026-03-28     25963.140625
9  2026-04-01     26037.423828
10 2026-04-05     26139.556641
11 2026-04-09     26097.412109
12 2026-04-13     26097.714844
13 2026-04-17     26103.210938
14 2026-04-21     26221.367188
15 2026-04-25     26233.923828
16 2026-04-29     26185.478516
17 2026-05-03     26053.662109
18 2026-05-07     26042.697266
19 2026-05-11     26144.277344
20 2026-05-15     26140.972656
21 2026-05-19     26065.414062
22 2026-05-23     25970.685547
23 2026-05-27     25975.070312
24 2026-05-31     26017.726562
25 2026-06-04     25985.625000
26 2026-06-08     25942.386719
27 2026-06-12     25943.136719
28 2026-06-16     26048.365234
29 2026-06-20     26144.626953


In [356]:
close_preds = future_prices.copy()

## For Open Price

In [357]:
# ============================================================
# 2️⃣ LOAD & PREPROCESS MULTI-YEAR NSE DATASET
# ============================================================

df = pd.read_csv("NIFTY_5_Years.csv", encoding="utf-8-sig")

# Clean columns
df.columns = df.columns.str.strip()

# Convert date
df['Date'] = pd.to_datetime(df['Date'])

# Sort ascending
df = df.sort_values("Date")

# Set index
df.set_index("Date", inplace=True)

# Use Close price
data = df[['Open']]

# Train-Test Split (80-20)
train_size = int(len(data) * 0.8)
train, test = data[:train_size], data[train_size:]

print("Training size:", len(train))
print("Testing size :", len(test))

Training size: 992
Testing size : 248


In [358]:
import numpy as np

def predict_future_prices(model, last_data, scaler, time_steps, future_days):
    
    # Ensure correct shape
    input_seq = last_data[-time_steps:]
    input_seq = input_seq.reshape(1, time_steps, 1)

    future_predictions = []

    for _ in range(future_days):
        
        pred = model.predict(input_seq, verbose=0)
        
        # Store prediction
        future_predictions.append(pred[0, 0])
        
        # Reshape prediction properly to (1,1,1)
        pred_reshaped = pred.reshape(1, 1, 1)
        
        # Remove first timestep and append prediction
        input_seq = np.concatenate(
            (input_seq[:, 1:, :], pred_reshaped),
            axis=1
        )

    # Convert back to original scale
    future_predictions = scaler.inverse_transform(
        np.array(future_predictions).reshape(-1, 1)
    )

    return future_predictions.flatten()

In [359]:
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

In [360]:
time_steps = 30

# --- SARIMA ---
sarima_model = SARIMAX(train['Open'],
                       order=(4,0,3),
                       seasonal_order=(1,0,1,5))

sarima_result = sarima_model.fit()

forecast = sarima_result.forecast(steps=len(test))

# --- Residuals ---
residuals = test['Open'].values - forecast.values
residuals = residuals.reshape(-1,1)

# --- Scaling ---
from sklearn.preprocessing import StandardScaler
scaler_res = StandardScaler()
res_scaled = scaler_res.fit_transform(residuals)

X_res, y_res = create_sequences(res_scaled, time_steps)

# --- RNN ---
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

model = Sequential([
    SimpleRNN(61, return_sequences=True, input_shape=(time_steps,1)),
    Dropout(0.2),
    SimpleRNN(32),
    Dense(1)
])

model.compile(optimizer=Adam(0.0005), loss='huber')

early_stop = EarlyStopping(patience=10, restore_best_weights=True)

model.fit(X_res, y_res,
          epochs=150,
          batch_size=16,
          callbacks=[early_stop],
          verbose=0)

# --- Prediction ---
pred = model.predict(X_res)
pred = scaler_res.inverse_transform(pred)

final_pred = forecast[time_steps:].values + pred.flatten()
actual = test['Open'].values[time_steps:]

7/7 [==============================] - 0s 3ms/step


In [361]:
future_days = y
future_prices = predict_future_prices(
    model,
    test_scaled,
    scaler,
    time_steps,
    future_days
)

print("Future Prices:", future_prices)

Future Prices: [25542.553 25726.639 26005.426 26260.033 26337.86  26133.39  26408.68
 26444.383 26559.209 26727.547 26738.13  26640.225 26533.078 26707.031
 26678.951 26659.797 26519.863 26382.992 26430.52  26389.672 26373.33
 26344.492 26231.57  26190.34  26262.602 26352.795 26392.82  26416.367
 26406.242 26471.822]


In [362]:
last_date = df.index[-1]
future_dates = pd.date_range(last_date, periods=future_days+1, freq=x)[1:]

future_df = pd.DataFrame({
    "Date": future_dates,
    "Predicted_Price": future_prices
})

print(future_df)

         Date  Predicted_Price
0  2026-02-24     25542.552734
1  2026-02-28     25726.638672
2  2026-03-04     26005.425781
3  2026-03-08     26260.033203
4  2026-03-12     26337.859375
5  2026-03-16     26133.390625
6  2026-03-20     26408.679688
7  2026-03-24     26444.382812
8  2026-03-28     26559.208984
9  2026-04-01     26727.546875
10 2026-04-05     26738.130859
11 2026-04-09     26640.224609
12 2026-04-13     26533.078125
13 2026-04-17     26707.031250
14 2026-04-21     26678.951172
15 2026-04-25     26659.796875
16 2026-04-29     26519.863281
17 2026-05-03     26382.992188
18 2026-05-07     26430.519531
19 2026-05-11     26389.671875
20 2026-05-15     26373.330078
21 2026-05-19     26344.492188
22 2026-05-23     26231.570312
23 2026-05-27     26190.339844
24 2026-05-31     26262.601562
25 2026-06-04     26352.794922
26 2026-06-08     26392.820312
27 2026-06-12     26416.367188
28 2026-06-16     26406.242188
29 2026-06-20     26471.822266


In [363]:
open_preds = future_prices.copy()

## For High

In [364]:
# ============================================================
# 2️⃣ LOAD & PREPROCESS MULTI-YEAR NSE DATASET
# ============================================================

df = pd.read_csv("NIFTY_5_Years.csv", encoding="utf-8-sig")

# Clean columns
df.columns = df.columns.str.strip()

# Convert date
df['Date'] = pd.to_datetime(df['Date'])

# Sort ascending
df = df.sort_values("Date")

# Set index
df.set_index("Date", inplace=True)

# Use Close price
data = df[['High']]

# Train-Test Split (80-20)
train_size = int(len(data) * 0.8)
train, test = data[:train_size], data[train_size:]

print("Training size:", len(train))
print("Testing size :", len(test))

Training size: 992
Testing size : 248


In [365]:
import numpy as np

def predict_future_prices(model, last_data, scaler, time_steps, future_days):
    
    # Ensure correct shape
    input_seq = last_data[-time_steps:]
    input_seq = input_seq.reshape(1, time_steps, 1)

    future_predictions = []

    for _ in range(future_days):
        
        pred = model.predict(input_seq, verbose=0)
        
        # Store prediction
        future_predictions.append(pred[0, 0])
        
        # Reshape prediction properly to (1,1,1)
        pred_reshaped = pred.reshape(1, 1, 1)
        
        # Remove first timestep and append prediction
        input_seq = np.concatenate(
            (input_seq[:, 1:, :], pred_reshaped),
            axis=1
        )

    # Convert back to original scale
    future_predictions = scaler.inverse_transform(
        np.array(future_predictions).reshape(-1, 1)
    )

    return future_predictions.flatten()

In [366]:
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

In [367]:
time_steps = 30

# --- SARIMA ---
sarima_model = SARIMAX(train['High'],
                       order=(4,0,3),
                       seasonal_order=(1,0,1,5))

sarima_result = sarima_model.fit()

forecast = sarima_result.forecast(steps=len(test))

# --- Residuals ---
residuals = test['High'].values - forecast.values
residuals = residuals.reshape(-1,1)

# --- Scaling ---
from sklearn.preprocessing import StandardScaler
scaler_res = StandardScaler()
res_scaled = scaler_res.fit_transform(residuals)

X_res, y_res = create_sequences(res_scaled, time_steps)

# --- RNN ---
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

model = Sequential([
    SimpleRNN(61, return_sequences=True, input_shape=(time_steps,1)),
    Dropout(0.2),
    SimpleRNN(32),
    Dense(1)
])

model.compile(optimizer=Adam(0.0005), loss='huber')

early_stop = EarlyStopping(patience=10, restore_best_weights=True)

model.fit(X_res, y_res,
          epochs=150,
          batch_size=16,
          callbacks=[early_stop],
          verbose=0)

# --- Prediction ---
pred = model.predict(X_res)
pred = scaler_res.inverse_transform(pred)

final_pred = forecast[time_steps:].values + pred.flatten()
actual = test['High'].values[time_steps:]

7/7 [==============================] - 0s 3ms/step


In [368]:
future_days = y
future_prices = predict_future_prices(
    model,
    test_scaled,
    scaler,
    time_steps,
    future_days
)

print("Future Prices:", future_prices)

Future Prices: [26244.73  26726.139 27185.834 27585.436 27993.11  28217.053 28538.533
 28733.777 28882.1   29031.977 28870.516 28632.246 28586.9   28479.135
 28270.043 28192.422 27985.904 27945.621 28057.379 28150.363 28279.824
 28357.082 28420.078 28596.281 28806.438 28867.746 28928.111 28958.07
 28886.492 28899.873]


In [369]:
last_date = df.index[-1]
future_dates = pd.date_range(last_date, periods=future_days+1, freq=x)[1:]

future_df = pd.DataFrame({
    "Date": future_dates,
    "Predicted_Price": future_prices
})

print(future_df)

         Date  Predicted_Price
0  2026-02-24     26244.730469
1  2026-02-28     26726.138672
2  2026-03-04     27185.833984
3  2026-03-08     27585.435547
4  2026-03-12     27993.109375
5  2026-03-16     28217.052734
6  2026-03-20     28538.533203
7  2026-03-24     28733.777344
8  2026-03-28     28882.099609
9  2026-04-01     29031.976562
10 2026-04-05     28870.515625
11 2026-04-09     28632.246094
12 2026-04-13     28586.900391
13 2026-04-17     28479.134766
14 2026-04-21     28270.042969
15 2026-04-25     28192.421875
16 2026-04-29     27985.904297
17 2026-05-03     27945.621094
18 2026-05-07     28057.378906
19 2026-05-11     28150.363281
20 2026-05-15     28279.824219
21 2026-05-19     28357.082031
22 2026-05-23     28420.078125
23 2026-05-27     28596.281250
24 2026-05-31     28806.437500
25 2026-06-04     28867.746094
26 2026-06-08     28928.111328
27 2026-06-12     28958.070312
28 2026-06-16     28886.492188
29 2026-06-20     28899.873047


In [370]:
high_preds = future_prices.copy()

## For Low

In [371]:
# ============================================================
# 2️⃣ LOAD & PREPROCESS MULTI-YEAR NSE DATASET
# ============================================================

df = pd.read_csv("NIFTY_5_Years.csv", encoding="utf-8-sig")

# Clean columns
df.columns = df.columns.str.strip()

# Convert date
df['Date'] = pd.to_datetime(df['Date'])

# Sort ascending
df = df.sort_values("Date")

# Set index
df.set_index("Date", inplace=True)

# Use Close price
data = df[['Low']]

# Train-Test Split (80-20)
train_size = int(len(data) * 0.8)
train, test = data[:train_size], data[train_size:]

print("Training size:", len(train))
print("Testing size :", len(test))

Training size: 992
Testing size : 248


In [372]:
import numpy as np

def predict_future_prices(model, last_data, scaler, time_steps, future_days):
    
    # Ensure correct shape
    input_seq = last_data[-time_steps:]
    input_seq = input_seq.reshape(1, time_steps, 1)

    future_predictions = []

    for _ in range(future_days):
        
        pred = model.predict(input_seq, verbose=0)
        
        # Store prediction
        future_predictions.append(pred[0, 0])
        
        # Reshape prediction properly to (1,1,1)
        pred_reshaped = pred.reshape(1, 1, 1)
        
        # Remove first timestep and append prediction
        input_seq = np.concatenate(
            (input_seq[:, 1:, :], pred_reshaped),
            axis=1
        )

    # Convert back to original scale
    future_predictions = scaler.inverse_transform(
        np.array(future_predictions).reshape(-1, 1)
    )

    return future_predictions.flatten()

In [373]:
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

In [374]:
time_steps = 30

# --- SARIMA ---
sarima_model = SARIMAX(train['Low'],
                       order=(4,0,3),
                       seasonal_order=(1,0,1,5))

sarima_result = sarima_model.fit()

forecast = sarima_result.forecast(steps=len(test))

# --- Residuals ---
residuals = test['Low'].values - forecast.values
residuals = residuals.reshape(-1,1)

# --- Scaling ---
from sklearn.preprocessing import StandardScaler
scaler_res = StandardScaler()
res_scaled = scaler_res.fit_transform(residuals)

X_res, y_res = create_sequences(res_scaled, time_steps)

# --- RNN ---
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

model = Sequential([
    SimpleRNN(61, return_sequences=True, input_shape=(time_steps,1)),
    Dropout(0.2),
    SimpleRNN(32),
    Dense(1)
])

model.compile(optimizer=Adam(0.0005), loss='huber')

early_stop = EarlyStopping(patience=10, restore_best_weights=True)

model.fit(X_res, y_res,
          epochs=150,
          batch_size=16,
          callbacks=[early_stop],
          verbose=0)

# --- Prediction ---
pred = model.predict(X_res)
pred = scaler_res.inverse_transform(pred)

final_pred = forecast[time_steps:].values + pred.flatten()
actual = test['Low'].values[time_steps:]

7/7 [==============================] - 0s 4ms/step


In [375]:
future_days = y
future_prices = predict_future_prices(
    model,
    test_scaled,
    scaler,
    time_steps,
    future_days
)

print("Future Prices:", future_prices)

Future Prices: [25307.578 25384.838 25441.234 25482.494 25598.691 25559.229 25527.443
 25725.168 25802.508 25737.854 25762.848 25733.094 25904.508 26045.85
 26148.852 26116.809 26007.44  26004.375 26059.344 26056.174 25957.537
 25878.334 25791.967 25775.76  25720.725 25676.92  25617.484 25583.213
 25668.027 25761.06 ]


In [376]:
last_date = df.index[-1]
future_dates = pd.date_range(last_date, periods=future_days+1, freq=x)[1:]

future_df = pd.DataFrame({
    "Date": future_dates,
    "Predicted_Price": future_prices
})

print(future_df)

         Date  Predicted_Price
0  2026-02-24     25307.578125
1  2026-02-28     25384.837891
2  2026-03-04     25441.234375
3  2026-03-08     25482.494141
4  2026-03-12     25598.691406
5  2026-03-16     25559.228516
6  2026-03-20     25527.443359
7  2026-03-24     25725.167969
8  2026-03-28     25802.507812
9  2026-04-01     25737.853516
10 2026-04-05     25762.847656
11 2026-04-09     25733.093750
12 2026-04-13     25904.507812
13 2026-04-17     26045.849609
14 2026-04-21     26148.851562
15 2026-04-25     26116.808594
16 2026-04-29     26007.439453
17 2026-05-03     26004.375000
18 2026-05-07     26059.343750
19 2026-05-11     26056.173828
20 2026-05-15     25957.537109
21 2026-05-19     25878.333984
22 2026-05-23     25791.966797
23 2026-05-27     25775.759766
24 2026-05-31     25720.724609
25 2026-06-04     25676.919922
26 2026-06-08     25617.484375
27 2026-06-12     25583.212891
28 2026-06-16     25668.027344
29 2026-06-20     25761.060547


In [377]:
low_preds = future_prices.copy()

## For Volume

In [378]:
# ============================================================
# 2️⃣ LOAD & PREPROCESS MULTI-YEAR NSE DATASET
# ============================================================

df = pd.read_csv("NIFTY_5_Years.csv", encoding="utf-8-sig")

# Clean columns
df.columns = df.columns.str.strip()

# Convert date
df['Date'] = pd.to_datetime(df['Date'])

# Sort ascending
df = df.sort_values("Date")

# Set index
df.set_index("Date", inplace=True)

# Use Close price
data = df[['Shares Traded']]

# Train-Test Split (80-20)
train_size = int(len(data) * 0.8)
train, test = data[:train_size], data[train_size:]

print("Training size:", len(train))
print("Testing size :", len(test))

Training size: 992
Testing size : 248


In [379]:
import numpy as np

def predict_future_prices(model, last_data, scaler, time_steps, future_days):
    
    # Ensure correct shape
    input_seq = last_data[-time_steps:]
    input_seq = input_seq.reshape(1, time_steps, 1)

    future_predictions = []

    for _ in range(future_days):
        
        pred = model.predict(input_seq, verbose=0)
        
        # Store prediction
        future_predictions.append(pred[0, 0])
        
        # Reshape prediction properly to (1,1,1)
        pred_reshaped = pred.reshape(1, 1, 1)
        
        # Remove first timestep and append prediction
        input_seq = np.concatenate(
            (input_seq[:, 1:, :], pred_reshaped),
            axis=1
        )

    # Convert back to original scale
    future_predictions = scaler.inverse_transform(
        np.array(future_predictions).reshape(-1, 1)
    )

    return future_predictions.flatten()

In [380]:
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

In [381]:
time_steps = 30

# --- SARIMA ---
sarima_model = SARIMAX(train['Shares Traded'],
                       order=(4,0,3),
                       seasonal_order=(1,0,1,5))

sarima_result = sarima_model.fit()

forecast = sarima_result.forecast(steps=len(test))

# --- Residuals ---
residuals = test['Shares Traded'].values - forecast.values
residuals = residuals.reshape(-1,1)

# --- Scaling ---
from sklearn.preprocessing import StandardScaler
scaler_res = StandardScaler()
res_scaled = scaler_res.fit_transform(residuals)

X_res, y_res = create_sequences(res_scaled, time_steps)

# --- RNN ---
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

model = Sequential([
    SimpleRNN(61, return_sequences=True, input_shape=(time_steps,1)),
    Dropout(0.2),
    SimpleRNN(32),
    Dense(1)
])

model.compile(optimizer=Adam(0.0005), loss='huber')

early_stop = EarlyStopping(patience=10, restore_best_weights=True)

model.fit(X_res, y_res,
          epochs=150,
          batch_size=16,
          callbacks=[early_stop],
          verbose=0)

# --- Prediction ---
pred = model.predict(X_res)
pred = scaler_res.inverse_transform(pred)

final_pred = forecast[time_steps:].values + pred.flatten()
actual = test['Shares Traded'].values[time_steps:]

7/7 [==============================] - 0s 3ms/step


In [382]:
future_days = y
future_prices = predict_future_prices(
    model,
    test_scaled,
    scaler,
    time_steps,
    future_days
)

print("Future Prices:", future_prices)

Future Prices: [ 4.6934314e+08  1.4841454e+08  3.0571142e+08  2.6346130e+08
  5.8070624e+08  9.0657536e+08  3.6334896e+08  4.2523424e+08
  4.9429715e+08  3.4010816e+08  6.4951661e+08  5.2730765e+08
  3.3854285e+08  4.0027773e+08  3.8370749e+08  3.6969024e+08
  2.5634973e+08  2.5442704e+08  4.1016147e+08  5.6794854e+08
 -5.6114660e+07  3.8245373e+08  8.2772819e+08  3.6089315e+08
  4.3281677e+08  1.5975734e+08  1.1998501e+08  1.9795960e+08
 -6.8620352e+07  8.4421504e+08]


In [383]:
last_date = df.index[-1]
future_dates = pd.date_range(last_date, periods=future_days+1, freq=x)[1:]

future_df = pd.DataFrame({
    "Date": future_dates,
    "Predicted_Price": future_prices
})

print(future_df)

         Date  Predicted_Price
0  2026-02-24      469343136.0
1  2026-02-28      148414544.0
2  2026-03-04      305711424.0
3  2026-03-08      263461296.0
4  2026-03-12      580706240.0
5  2026-03-16      906575360.0
6  2026-03-20      363348960.0
7  2026-03-24      425234240.0
8  2026-03-28      494297152.0
9  2026-04-01      340108160.0
10 2026-04-05      649516608.0
11 2026-04-09      527307648.0
12 2026-04-13      338542848.0
13 2026-04-17      400277728.0
14 2026-04-21      383707488.0
15 2026-04-25      369690240.0
16 2026-04-29      256349728.0
17 2026-05-03      254427040.0
18 2026-05-07      410161472.0
19 2026-05-11      567948544.0
20 2026-05-15      -56114660.0
21 2026-05-19      382453728.0
22 2026-05-23      827728192.0
23 2026-05-27      360893152.0
24 2026-05-31      432816768.0
25 2026-06-04      159757344.0
26 2026-06-08      119985008.0
27 2026-06-12      197959600.0
28 2026-06-16      -68620352.0
29 2026-06-20      844215040.0


In [384]:
volume_preds = future_prices.copy()

# For Turnover

In [385]:
# ============================================================
# 2️⃣ LOAD & PREPROCESS MULTI-YEAR NSE DATASET
# ============================================================

df = pd.read_csv("NIFTY_5_Years.csv", encoding="utf-8-sig")

# Clean columns
df.columns = df.columns.str.strip()

# Convert date
df['Date'] = pd.to_datetime(df['Date'])

# Sort ascending
df = df.sort_values("Date")

# Set index
df.set_index("Date", inplace=True)

# Use Close price
data = df[['Turnover (₹ Cr)']]

# Train-Test Split (80-20)
train_size = int(len(data) * 0.8)
train, test = data[:train_size], data[train_size:]

print("Training size:", len(train))
print("Testing size :", len(test))

Training size: 992
Testing size : 248


In [386]:
import numpy as np

def predict_future_prices(model, last_data, scaler, time_steps, future_days):
    
    # Ensure correct shape
    input_seq = last_data[-time_steps:]
    input_seq = input_seq.reshape(1, time_steps, 1)

    future_predictions = []

    for _ in range(future_days):
        
        pred = model.predict(input_seq, verbose=0)
        
        # Store prediction
        future_predictions.append(pred[0, 0])
        
        # Reshape prediction properly to (1,1,1)
        pred_reshaped = pred.reshape(1, 1, 1)
        
        # Remove first timestep and append prediction
        input_seq = np.concatenate(
            (input_seq[:, 1:, :], pred_reshaped),
            axis=1
        )

    # Convert back to original scale
    future_predictions = scaler.inverse_transform(
        np.array(future_predictions).reshape(-1, 1)
    )

    return future_predictions.flatten()

In [387]:
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

In [388]:
time_steps = 30

# --- SARIMA ---
sarima_model = SARIMAX(train['Turnover (₹ Cr)'],
                       order=(4,0,3),
                       seasonal_order=(1,0,1,5))

sarima_result = sarima_model.fit()

forecast = sarima_result.forecast(steps=len(test))

# --- Residuals ---
residuals = test['Turnover (₹ Cr)'].values - forecast.values
residuals = residuals.reshape(-1,1)

# --- Scaling ---
from sklearn.preprocessing import StandardScaler
scaler_res = StandardScaler()
res_scaled = scaler_res.fit_transform(residuals)

X_res, y_res = create_sequences(res_scaled, time_steps)

# --- RNN ---
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

model = Sequential([
    SimpleRNN(61, return_sequences=True, input_shape=(time_steps,1)),
    Dropout(0.2),
    SimpleRNN(32),
    Dense(1)
])

model.compile(optimizer=Adam(0.0005), loss='huber')

early_stop = EarlyStopping(patience=10, restore_best_weights=True)

model.fit(X_res, y_res,
          epochs=150,
          batch_size=16,
          callbacks=[early_stop],
          verbose=0)

# --- Prediction ---
pred = model.predict(X_res)
pred = scaler_res.inverse_transform(pred)

final_pred = forecast[time_steps:].values + pred.flatten()
actual = test['Turnover (₹ Cr)'].values[time_steps:]

7/7 [==============================] - 0s 3ms/step


In [389]:
future_days = y
future_prices = predict_future_prices(
    model,
    test_scaled,
    scaler,
    time_steps,
    future_days
)

print("Future Prices:", future_prices)

Future Prices: [   2546.2493  -12363.48    -24762.438   -30883.363    -8893.439
  -40019.223   -45146.844   -23722.68   -107094.664  -106022.875
 -114409.984  -146114.58    -75629.6    -101763.85   -116743.45
  -93340.66    -79051.11    -73378.3     -63099.35    -70862.32
  -93151.375   -84287.3     -20749.861    18464.174    32181.682
   -1386.3746  -26538.95     -6291.2583   13036.953    73331.25  ]


In [390]:
last_date = df.index[-1]
future_dates = pd.date_range(last_date, periods=future_days+1, freq=x)[1:]

future_df = pd.DataFrame({
    "Date": future_dates,
    "Predicted_Price": future_prices
})

print(future_df)

         Date  Predicted_Price
0  2026-02-24      2546.249268
1  2026-02-28    -12363.480469
2  2026-03-04    -24762.437500
3  2026-03-08    -30883.363281
4  2026-03-12     -8893.439453
5  2026-03-16    -40019.222656
6  2026-03-20    -45146.843750
7  2026-03-24    -23722.679688
8  2026-03-28   -107094.664062
9  2026-04-01   -106022.875000
10 2026-04-05   -114409.984375
11 2026-04-09   -146114.578125
12 2026-04-13    -75629.601562
13 2026-04-17   -101763.851562
14 2026-04-21   -116743.453125
15 2026-04-25    -93340.656250
16 2026-04-29    -79051.109375
17 2026-05-03    -73378.296875
18 2026-05-07    -63099.351562
19 2026-05-11    -70862.320312
20 2026-05-15    -93151.375000
21 2026-05-19    -84287.296875
22 2026-05-23    -20749.861328
23 2026-05-27     18464.173828
24 2026-05-31     32181.681641
25 2026-06-04     -1386.374634
26 2026-06-08    -26538.949219
27 2026-06-12     -6291.258301
28 2026-06-16     13036.953125
29 2026-06-20     73331.250000


In [391]:
turnover_preds = future_prices.copy()

In [392]:
from tabulate import tabulate

# Create DataFrame
future_df = pd.DataFrame({
    "Date": future_dates,
    "Open": open_preds,
    "High": high_preds,
    "Low": low_preds,
    "Close": close_preds,
    "Shares Traded": volume_preds,
    #"Turnover": turnover_preds
})

print(tabulate(future_df, headers='keys', tablefmt='pretty', showindex=False))

+---------------------+-----------------+-----------------+-----------------+-----------------+---------------+
|        Date         |      Open       |      High       |       Low       |      Close      | Shares Traded |
+---------------------+-----------------+-----------------+-----------------+-----------------+---------------+
| 2026-02-24 00:00:00 | 25542.552734375 | 26244.73046875  |  25307.578125   |  25643.4453125  |  469343136.0  |
| 2026-02-28 00:00:00 | 25726.638671875 | 26726.138671875 | 25384.837890625 |  25762.7265625  |  148414544.0  |
| 2026-03-04 00:00:00 | 26005.42578125  | 27185.833984375 |  25441.234375   |  25770.671875   |  305711424.0  |
| 2026-03-08 00:00:00 | 26260.033203125 | 27585.435546875 | 25482.494140625 | 25703.720703125 |  263461296.0  |
| 2026-03-12 00:00:00 |  26337.859375   |  27993.109375   | 25598.69140625  | 25872.716796875 |  580706240.0  |
| 2026-03-16 00:00:00 |  26133.390625   | 28217.052734375 | 25559.228515625 | 26046.423828125 |  9065753